In [ ]:

!pip -q install pandas numpy scipy matplotlib

from pathlib import Path

DRIVE_PROJECT_DIR = Path("MyDrive/GenderBias")


CREATE_ZIP = False

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = Path("/content/drive") / DRIVE_PROJECT_DIR
    print(f"Google Drive mounted: {PROJECT_DIR}")
except (ImportError, ModuleNotFoundError):
    
    PROJECT_DIR = Path.cwd()
    print(f"Not in Colab — using local project folder: {PROJECT_DIR}")

INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_DIR = PROJECT_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HUMAN_RAW_PATH = INPUT_DIR / "HumanDataRaw.csv"
MODEL_ANNOTATIONS_PATH = INPUT_DIR / "parsed_annotations.csv"
HUMAN_AGGREGATE_PATH = INPUT_DIR / "HumanAnnotation.csv"  

required_inputs = [HUMAN_RAW_PATH, MODEL_ANNOTATIONS_PATH]
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(
        "Missing required input file(s). Upload them to the Google Drive input "
        "folder shown above, then rerun this cell:\n- " + "\n- ".join(missing_inputs)
    )

print(f"Input folder:  {INPUT_DIR}")
print(f"Output folder: {OUTPUT_DIR}")


## Core functions

In [ ]:
import numpy as np, pandas as pd, itertools, re
from collections import Counter
from scipy import stats

CATS = ['RB', 'LEB', 'M', 'NB']
BIAS = ['RB', 'LEB', 'M']
BIASSET = set(BIAS)
SEED = 2026

def is_contradictory(s):
    """A label set pairing No Bias with at least one bias category."""
    return s is not None and 'NB' in s and bool(s & BIASSET)


def load_human_raw(path='HumanDataRaw.csv'):
    """Wide export: one row per sentence, one column per annotator.
    Sentence numbering RESTARTS at 1 in each article, so article boundaries are
    recovered from the resets; sentence numbers may end in a full-width period."""
    r = pd.read_csv(path, encoding='utf-8-sig')
    r.columns = ['sent'] + [f'A{i}' for i in range(1, 31)]
    r['num'] = r.sent.astype(str).str.extract(r'^\s*(\d+)\s*[.\uff0e\u3002]')[0].astype(int)
    art, a = [], 0
    for n in r.num:
        if n == 1:
            a += 1
        art.append(f'P{a}')
    r['article'] = art
    r['S'] = r.article + '_' + r.num.astype(str)
    assert r.S.nunique() == 57, "expected 57 unique sentence keys"
    assert list(Counter(r.article).values()) == [20, 19, 18], "unexpected article lengths"
    cells = {}
    for _, row in r.iterrows():
        for i in range(1, 31):
            cells[(row.S, i)] = frozenset(
                t for t in re.split(r'[^A-Za-z]+', str(row[f'A{i}'])) if t in CATS)
    return r, cells, list(r.S), list(range(1, 31))

def load_models(path='parsed_annotations.csv'):
    d = pd.read_csv(path)
    d['S'] = d.article.astype(str) + '_' + d.sentence_id.astype(str)
    return d

def build_repetition_table(d):
    """One row per model x repetition x sentence after duplicate-row unioning."""
    rows = []
    key = ['model_key', 'rep', 'S']
    for (model, rep, sent), g in d.groupby(key, sort=True):
        label_set = set()
        for value in g.labels:
            label_set |= {x for x in str(value).split('+') if x in CATS}
        first = g.iloc[0]
        rows.append({
            'model_key': model,
            'model_string': first.get('model_string', ''),
            'provider': first.get('provider', ''),
            'rep': int(rep),
            'article': first.article,
            'sentence_id': int(first.sentence_id),
            'S': sent,
            'labels': '+'.join(c for c in CATS if c in label_set),
            'label_set': frozenset(label_set),
            'source_rows': len(g),
            'contradictory': is_contradictory(label_set),
        })
    return pd.DataFrame(rows)

def clean_human(cells):
    """Delete contradictory judgments; deleted cells become None."""
    out, deleted = {}, []
    for k, s in cells.items():
        if is_contradictory(s):
            out[k] = None
            deleted.append(k)
        else:
            out[k] = s
    return out, deleted

def consolidate(d):
    """Apply the model coding rule. cons values may be None (NA)."""
    reps = {}
    for (m, r, s), g in d.groupby(['model_key', 'rep', 'S']):
        labels = set()
        for x in g.labels:
            labels |= set(str(x).split('+'))
        reps[(m, r, s)] = frozenset(labels)
    models, sents = sorted(d.model_key.unique()), sorted(d.S.unique())
    cons = {}
    log = {'rep_deleted': [], 'na_cells': [], 'n_survivors': Counter(),
           'single_survivor': [], 'synthesised': 0}
    for m in models:
        for s in sents:
            trip = [reps[(m, r, s)] for r in (1, 2, 3)]
            for r, st in zip((1, 2, 3), trip):
                if is_contradictory(st):
                    log['rep_deleted'].append((m, r, s, st))
            surv = [st for st in trip if not is_contradictory(st)]
            log['n_survivors'][len(surv)] += 1
            if len(surv) == 0:
                out = None
                log['na_cells'].append((m, s, 'no surviving repetition'))
            elif len(surv) == 1:
                out = surv[0]
                log['single_survivor'].append((m, s))
            else:
                counts = Counter(label for st in surv for label in st)
                out = frozenset(label for label, n in counts.items() if n >= 2)
                if not out:
                    out = None
                    log['na_cells'].append((m, s, 'no code in two surviving repetitions'))
            if out is not None and out not in surv:
                log['synthesised'] += 1
            cons[(m, s)] = out
    return cons, models, sents, reps, log


def human_vec(cells, order, anns, cat=None):
    """(selected, n_valid) per sentence. cat=None means any bias category."""
    selected, n_valid = [], []
    for s in order:
        values = [cells[(s, a)] for a in anns if cells[(s, a)] is not None]
        n_valid.append(len(values))
        selected.append(sum(
            1 for x in values if (cat in x if cat else bool(x - {'NB'}))
        ))
    return np.asarray(selected), np.asarray(n_valid)

def model_vec(cons, models, order, cat=None):
    selected, n_valid = [], []
    for s in order:
        values = [cons[(m, s)] for m in models if cons[(m, s)] is not None]
        n_valid.append(len(values))
        selected.append(sum(
            1 for x in values if (cat in x if cat else x != frozenset({'NB'}))
        ))
    return np.asarray(selected), np.asarray(n_valid)

def repetition_vec(reps, models, order, rep, cat=None):
    """Unconsolidated model vector after deleting contradictory repetitions."""
    selected, n_valid = [], []
    for s in order:
        values = [
            reps[(m, rep, s)] for m in models
            if not is_contradictory(reps[(m, rep, s)])
        ]
        n_valid.append(len(values))
        selected.append(sum(
            1 for x in values if (cat in x if cat else x != frozenset({'NB'}))
        ))
    return np.asarray(selected), np.asarray(n_valid)

def stability_after_cleaning(reps, model, order):
    """Exact agreement across all surviving repetitions, among comparable cells."""
    stable = comparable = 0
    for s in order:
        values = [
            reps[(model, rep, s)] for rep in (1, 2, 3)
            if not is_contradictory(reps[(model, rep, s)])
        ]
        if len(values) >= 2:
            comparable += 1
            stable += int(len(set(values)) == 1)
    return stable, comparable


def fleiss_kappa(sel, n_i):
    """Binary Fleiss' kappa admitting a different rater count per item."""
    sel, n_i = np.asarray(sel, float), np.asarray(n_i, float)
    keep = n_i >= 2
    sel, n_i = sel[keep], n_i[keep]
    if len(sel) == 0 or sel.sum() == 0 or (sel == n_i).all():
        return np.nan
    Pi = ((sel**2 + (n_i - sel)**2) - n_i) / (n_i * (n_i - 1))
    p = sel.sum() / n_i.sum()
    Pe = p**2 + (1 - p)**2
    return (Pi.mean() - Pe) / (1 - Pe) if Pe < 1 else np.nan

def pairwise_agreement(sel, n_i):
    """Mean non-chance-corrected pair agreement; prevalence-dependent."""
    sel, n_i = np.asarray(sel, float), np.asarray(n_i, float)
    keep = n_i >= 2
    sel, n_i = sel[keep], n_i[keep]
    agree = sel * (sel - 1) / 2 + (n_i - sel) * (n_i - sel - 1) / 2
    return (agree / (n_i * (n_i - 1) / 2)).mean()

def krippendorff_alpha(sel, n_i):
    """Nominal alpha for binary data with variable coders per item."""
    sel, n_i = np.asarray(sel, float), np.asarray(n_i, float)
    keep = n_i >= 2
    sel, n_i = sel[keep], n_i[keep]
    ntot, selected = n_i.sum(), sel.sum()
    Do = (2 * sel * (n_i - sel) / (n_i - 1)).sum() / ntot
    De = 2 * selected * (ntot - selected) / (ntot * (ntot - 1))
    return 1 - Do / De if De > 0 else np.nan

def bootstrap_kappa_ci(sel, n_i, B=5000, seed=SEED):
    """Percentile interval from resampling sentences."""
    rng = np.random.default_rng(seed)
    sel, n_i = np.asarray(sel), np.asarray(n_i)
    N = len(sel)
    out = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, N, N)
        out[b] = fleiss_kappa(sel[idx], n_i[idx])
    out = out[np.isfinite(out)]
    return np.percentile(out, 2.5), np.percentile(out, 97.5)

def bootstrap_mean_ci(values, B=5000, seed=SEED):
    """Sentence-level percentile interval for a mean."""
    values = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(values), size=(B, len(values)))
    means = values[idx].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

def bootstrap_difference_ci(x, y, B=5000, seed=SEED):
    """Paired sentence bootstrap for mean(x)-mean(y)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    assert len(x) == len(y)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(B, len(x)))
    diffs = (x[idx] - y[idx]).mean(axis=1)
    return np.percentile(diffs, [2.5, 97.5])

def annotator_bootstrap(cells, order, anns, cat=None, n_sub=6, B=10000, seed=SEED):
    """Kappa from random subsets of real annotators, preserving profiles."""
    rng = np.random.default_rng(seed)
    annotators = np.asarray(anns)
    out = np.empty(B)
    for b in range(B):
        sub = rng.choice(annotators, n_sub, replace=False)
        selected, n_valid = [], []
        for s in order:
            values = [cells[(s, a)] for a in sub if cells[(s, a)] is not None]
            n_valid.append(len(values))
            selected.append(sum(
                1 for x in values if (cat in x if cat else bool(x - {'NB'}))
            ))
        out[b] = fleiss_kappa(selected, n_valid)
    return out[np.isfinite(out)]

def annotator_profile(cells, order, anns):
    rows = []
    for a in anns:
        sets = [cells[(s, a)] for s in order if cells[(s, a)] is not None]
        flagged = sum(1 for x in sets if x - {'NB'})
        rows.append({
            'annotator': a,
            'valid_sentences': len(sets),
            'flagged': flagged,
            'pct_flagged': round(100 * flagged / len(sets), 1),
            **{c: sum(1 for x in sets if c in x) for c in CATS},
            'labels': sum(len(x) for x in sets),
        })
    return pd.DataFrame(rows).set_index('annotator')

def concordance(x, y):
    """P(models order a human-ordered pair the same way), model ties = 0.5."""
    n = len(x)
    conc = disc = tie = 0
    for i, j in itertools.combinations(range(n), 2):
        dx = x[i] - x[j]
        if dx == 0:
            continue
        dy = y[i] - y[j]
        if dy == 0:
            tie += 1
        elif np.sign(dx) == np.sign(dy):
            conc += 1
        else:
            disc += 1
    total = conc + disc + tie
    value = (conc + 0.5 * tie) / total if total else np.nan
    return value, conc, disc, tie, total

def ordering_metrics(human, model):
    """Three ordering statistics used throughout sensitivity analyses."""
    rho, p_rho = stats.spearmanr(human, model)
    tau, p_tau = stats.kendalltau(human, model, variant='b')
    conc = concordance(human, model)[0]
    return {
        'rho': float(rho),
        'rho_p': float(p_rho),
        'tau_b': float(tau),
        'tau_p': float(p_tau),
        'concordance': float(conc),
    }

def concordance_bootstrap_ci(human, model, B=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    human, model = np.asarray(human), np.asarray(model)
    values = []
    for _ in range(B):
        idx = rng.integers(0, len(human), len(human))
        value = concordance(human[idx], model[idx])[0]
        if np.isfinite(value):
            values.append(value)
    return np.percentile(values, [2.5, 97.5])

def threshold_fit_binomial(human_prop, model_selected, model_n):
    """Descriptive binomial logit using per-sentence model counts."""
    hp = np.asarray(human_prop, float)
    selected = np.asarray(model_selected, float)
    n_i = np.asarray(model_n, float)
    keep = n_i > 0
    hp, selected, n_i = hp[keep], selected[keep], n_i[keep]
    X = np.column_stack([np.ones_like(hp), hp])
    beta = np.zeros(2)
    for _ in range(200):
        eta = np.clip(X @ beta, -35, 35)
        mu = 1 / (1 + np.exp(-eta))
        score = X.T @ (selected - n_i * mu)
        information = (X * (n_i * mu * (1 - mu))[:, None]).T @ X
        try:
            step = np.linalg.solve(information, score)
        except np.linalg.LinAlgError:
            return np.array([np.nan, np.nan]), np.nan
        beta += step
        if np.max(np.abs(step)) < 1e-10:
            break
    threshold = -beta[0] / beta[1] if beta[1] != 0 else np.nan
    return beta, threshold

def fitted_probability(beta, value):
    if not np.isfinite(beta).all():
        return np.nan
    eta = np.clip(beta[0] + beta[1] * value, -35, 35)
    return float(1 / (1 + np.exp(-eta)))

def threshold_diagnostics(beta, threshold, observed_human):
    lo, hi = float(np.min(observed_human)), float(np.max(observed_human))
    reached = bool(np.isfinite(threshold) and lo <= threshold <= hi)
    return {
        'threshold_estimate': float(threshold) if np.isfinite(threshold) else np.nan,
        'within_observed_range': reached,
        'observed_human_min': lo,
        'observed_human_max': hi,
        'fitted_p_at_observed_max': fitted_probability(beta, hi),
    }


## Load and apply the coding rules

In [ ]:
import numpy as np, pandas as pd, itertools, json
from collections import Counter
from scipy import 

raw, cells_raw, order, anns = load_human_raw(HUMAN_RAW_PATH)
cells, deleted = clean_human(cells_raw)
d = load_models(MODEL_ANNOTATIONS_PATH)
rep_table = build_repetition_table(d)
rep_clean = rep_table.loc[~rep_table.contradictory].copy()
cons, models, sents, reps, log = consolidate(d)

assert set(order) == set(sents)
assert (d.groupby(['model_key', 'rep', 'article']).sentence_id.nunique()
          .unstack()[['P1', 'P2', 'P3']].values == [20, 19, 18]).all()
assert not any(is_contradictory(v) for v in cons.values()), "contradictory consolidated model output"
assert not any(is_contradictory(v) for v in cells.values()), "contradictory human judgment"
assert len(rep_table) == len(models) * 3 * len(order), "unexpected repetition-cell count"
assert len(rep_clean) == len(rep_table) - len(log['rep_deleted'])
assert not rep_clean.contradictory.any()

print("HUMAN")
print(f"  judgments deleted: {len(deleted)} of {len(cells_raw)} "
      f"({100 * len(deleted) / len(cells_raw):.2f}%), across "
      f"{len({a for _, a in deleted})} annotators and "
      f"{len({s for s, _ in deleted})} sentences")
hsel, hn = human_vec(cells, order, anns)
hum_prop = hsel / hn
print(f"  valid raters per sentence: {hn.min()}-{hn.max()}; "
      f"{(hn < 30).sum()} sentences below 30")

print("\nMODEL")
print(f"  raw parsed rows: {len(d)}")
print(f"  unique repetition cells after duplicate union: {len(rep_table)}")
print(f"  repetitions deleted: {len(log['rep_deleted'])}")
for m, rep, s, labels in log['rep_deleted']:
    print(f"     {m} rep{rep} {s}: {{{','.join(sorted(labels))}}}")
print(f"  surviving repetition cells used for sensitivity: {len(rep_clean)}")
print(f"  surviving repetitions per consolidated cell: {dict(log['n_survivors'])}")
print(f"  cells resolved from a single survivor: {len(log['single_survivor'])}")
print(f"  cells returned as NA: {len(log['na_cells'])}")
for m, s, why in log['na_cells']:
    print(f"     {m} {s}: {why}")
print(f"  consolidated sets matching no surviving repetition: {log['synthesised']}")

msel, mn = model_vec(cons, models, order)
mod_prop = msel / mn

try:
    agg = pd.read_csv(HUMAN_AGGREGATE_PATH, encoding='utf-8-sig')
    agg.columns = ['sid', 'en', 'zh', 'RB', 'LEB', 'M', 'NB', 'total', 'n_ann']
    agg['idx'] = agg.sid.str.extract(r'(\d+)')[0].astype(int)
    offsets = {'P1': 0, 'P2': 20, 'P3': 39}
    key = {s: raw.num[i] + offsets[raw.article[i]] for i, s in enumerate(order)}
    mismatch_counts = {}
    aggregate = agg.set_index('idx')
    for category in CATS:
        raw_counts = np.asarray([
            sum(1 for annotator in anns if category in cells_raw[(sent, annotator)])
            for sent in order
        ])
        aggregate_counts = aggregate[category].reindex(
            [key[sent] for sent in order]
        ).to_numpy()
        mismatch_counts[category] = int(np.count_nonzero(
            raw_counts != aggregate_counts
        ))
    bad = sum(mismatch_counts.values())
    print(f"\ncross-check of uncleaned counts vs aggregate file: {bad} mismatches")
except FileNotFoundError:
    print("\naggregate file not supplied — cross-check skipped")

REPORT = {
    'seed': SEED,
    'human_deleted': len(deleted),
    'model_reps_deleted': len(log['rep_deleted']),
    'model_repetition_cells_raw': len(rep_table),
    'model_repetition_cells_clean': len(rep_clean),
    'model_na': len(log['na_cells']),
    'synthesised': log['synthesised'],
}


## Detection and category profiles



In [ ]:
rows = []
for model in models:
    values = [cons[(model, s)] for s in order if cons[(model, s)] is not None]
    labels = [label for value in values for label in value]
    bias_labels = sum(1 for label in labels if label != 'NB')
    flagged = sum(1 for value in values if value != frozenset({'NB'}))
    stable, comparable = stability_after_cleaning(reps, model, order)
    rows.append({
        'model': model,
        'sentences': len(values),
        'labels': len(labels),
        'bias_label_share': bias_labels / len(labels),
        'sentences_flagged': flagged,
        'sentence_flag_share': flagged / len(values),
        'stable_cells': stable,
        'stability_denominator': comparable,
        'stability_share': stable / comparable,
    })

human_label_total = sum(
    len(cells[(s, a)]) for s in order for a in anns if cells[(s, a)] is not None
)
human_bias_labels = sum(
    len(cells[(s, a)] - {'NB'})
    for s in order for a in anns if cells[(s, a)] is not None
)
rows.append({
    'model': 'HUMAN (aggregate)',
    'sentences': int(hn.sum()),
    'labels': human_label_total,
    'bias_label_share': human_bias_labels / human_label_total,
    'sentences_flagged': int(hsel.sum()),
    'sentence_flag_share': hum_prop.mean(),
    'stable_cells': np.nan,
    'stability_denominator': np.nan,
    'stability_share': np.nan,
})
T2_model = pd.DataFrame(rows)
print(T2_model.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

rows = []
for category in BIAS + [None]:
    name = category or 'Any bias'
    hs, hnn = human_vec(cells, order, anns, category)
    ms, mnn = model_vec(cons, models, order, category)
    hp, mp = hs / hnn, ms / mnn
    hlo, hhi = bootstrap_mean_ci(hp)
    mlo, mhi = bootstrap_mean_ci(mp)
    dlo, dhi = bootstrap_difference_ci(hp, mp)
    rows.append({
        'category': name,
        'human_sentence_share': hp.mean(),
        'human_95%_CI': f'[{hlo:.3f}, {hhi:.3f}]',
        'model_sentence_share': mp.mean(),
        'model_95%_CI': f'[{mlo:.3f}, {mhi:.3f}]',
        'human_minus_model': (hp - mp).mean(),
        'difference_95%_CI': f'[{dlo:.3f}, {dhi:.3f}]',
    })
T2_category = pd.DataFrame(rows)
print("\nCategory-specific sentence shares:")
print(T2_category.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

human_all_counts = {
    c: int(human_vec(cells, order, anns, c)[0].sum()) for c in CATS
}
model_all_counts = {
    c: int(model_vec(cons, models, order, c)[0].sum()) for c in CATS
}
rows = []
for track, counts in [('human', human_all_counts), ('model', model_all_counts)]:
    total = sum(counts.values())
    for category in CATS:
        rows.append({
            'track': track,
            'category': category,
            'assigned_label_count': counts[category],
            'all_assigned_labels': total,
            'label_share': counts[category] / total,
        })
T2_label_share = pd.DataFrame(rows)
print("\nCategory label shares (secondary):")
print(T2_label_share.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

human_counts = {c: human_all_counts[c] for c in BIAS}
model_counts = {c: model_all_counts[c] for c in BIAS}
rows = []
for track, counts in [('human', human_counts), ('model', model_counts)]:
    total = sum(counts.values())
    for category in BIAS:
        rows.append({
            'track': track,
            'category': category,
            'bias_label_count': counts[category],
            'within_bias_label_share': counts[category] / total,
        })
T2_label_profile = pd.DataFrame(rows)
print("\nWithin-bias label profile (descriptive):")
print(T2_label_profile.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\nModel co-occurrence:",
      Counter('+'.join(sorted(v)) for v in cons.values() if v).most_common())

REPORT['detection'] = {
    'model_summary': T2_model.to_dict('records'),
    'category_sentence_shares': T2_category.to_dict('records'),
    'category_label_shares': T2_label_share.to_dict('records'),
    'within_bias_label_profile': T2_label_profile.to_dict('records'),
}


## Agreement and the rater-matched null

In [ ]:
rows = []
for category in BIAS + [None]:
    name = category or 'Any bias / No Bias'
    hs, hnn = human_vec(cells, order, anns, category)
    ms, mnn = model_vec(cons, models, order, category)
    hlo, hhi = bootstrap_kappa_ci(hs, hnn)
    mlo, mhi = bootstrap_kappa_ci(ms, mnn)
    rows.append({
        'category': name,
        'human_k': fleiss_kappa(hs, hnn),
        'human_95%_CI': f'[{hlo:.3f}, {hhi:.3f}]',
        'human_pairwise': pairwise_agreement(hs, hnn),
        'human_alpha': krippendorff_alpha(hs, hnn),
        'model_k': fleiss_kappa(ms, mnn),
        'model_95%_CI': f'[{mlo:.3f}, {mhi:.3f}]',
        'model_pairwise': pairwise_agreement(ms, mnn),
        'model_alpha': krippendorff_alpha(ms, mnn),
    })
T3 = pd.DataFrame(rows)
print(T3.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\n--- model kappa located among random six-annotator human panels ---")
rows = []
for category in [None] + BIAS:
    name = category or 'Any bias / No Bias'
    six_humans = annotator_bootstrap(cells, order, anns, category, 6, B=10000)
    ms, mnn = model_vec(cons, models, order, category)
    model_k = fleiss_kappa(ms, mnn)
    rows.append({
        'category': name,
        'model_k': model_k,
        'median_k_six_humans': np.median(six_humans),
        '95%_interval_six_humans':
            f'[{np.percentile(six_humans, 2.5):.3f}, '
            f'{np.percentile(six_humans, 97.5):.3f}]',
        'model_percentile': (six_humans < model_k).mean() * 100,
    })
T4 = pd.DataFrame(rows)
print(T4.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

REPORT['agreement'] = T3.to_dict('records')
REPORT['agreement_matched'] = T4.to_dict('records')


## Annotator-level heterogeneity

In [ ]:
T5=annotator_profile(cells,order,anns)
print(T5.to_string())
print(f"\nflag rate {T5.pct_flagged.min()}-{T5.pct_flagged.max()}%, median {T5.pct_flagged.median()}%, "
      f"IQR {T5.pct_flagged.quantile(.25)}-{T5.pct_flagged.quantile(.75)}")
print(f"labels per annotator {T5.labels.min()}-{T5.labels.max()}; "
      f"valid sentences {T5.valid_sentences.min()}-{T5.valid_sentences.max()}")
print(f"M: {(T5.M==0).sum()} annotators never used it; max {T5.M.max()}; median {T5.M.median()}")
REPORT['annotator_flag_rate']={'min':float(T5.pct_flagged.min()),
    'median':float(T5.pct_flagged.median()),'max':float(T5.pct_flagged.max())}

## Level, ordering, threshold, and heterogeneity


In [ ]:
overall = ordering_metrics(hum_prop, mod_prop)
conc_value, conc_n, disc_n, tie_n, ordered_n = concordance(hum_prop, mod_prop)
conc_lo, conc_hi = concordance_bootstrap_ci(hum_prop, mod_prop)
tied_pairs = sum(
    1 for i, j in itertools.combinations(range(len(order)), 2)
    if mod_prop[i] == mod_prop[j]
)
total_pairs = len(order) * (len(order) - 1) // 2
b, threshold = threshold_fit_binomial(hum_prop, msel, mn)
threshold_info = threshold_diagnostics(b, threshold, hum_prop)

T6_overall = pd.DataFrame([{
    'human_mean_flag_share': hum_prop.mean(),
    'model_mean_flag_share': mod_prop.mean(),
    'human_model_ratio': hum_prop.mean() / mod_prop.mean(),
    'model_distinct_values': len(set(mod_prop)),
    'model_tied_pair_share': tied_pairs / total_pairs,
    **overall,
    'concordance_95%_CI': f'[{conc_lo:.3f}, {conc_hi:.3f}]',
    'concordant_pairs': conc_n,
    'discordant_pairs': disc_n,
    'model_tied_ordered_pairs': tie_n,
    'human_ordered_pairs': ordered_n,
    **threshold_info,
}])
print(T6_overall.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

if threshold_info['within_observed_range']:
    print(f"\nFitted P(model flag)=.50 at human endorsement {threshold:.3f}, "
          "within the observed range.")
else:
    print(f"\nFitted P(model flag) did not reach .50 in the observed human range "
          f"[{threshold_info['observed_human_min']:.3f}, "
          f"{threshold_info['observed_human_max']:.3f}].")
    print(f"At the maximum observed endorsement, fitted P(model flag)="
          f"{threshold_info['fitted_p_at_observed_max']:.3f}. "
          f"The algebraic crossing ({threshold:.3f}) is extrapolative.")

for value in (.25, .50, .75, threshold_info['observed_human_max']):
    print(f"   human={value:.3f} -> P(model flags)={fitted_probability(b, value):.3f}")


rows = []
for model in models:
    selected, n_valid = model_vec(cons, [model], order)
    model_prop = selected / n_valid
    rows.append({
        'model': model,
        'mean_flag_share': model_prop.mean(),
        **ordering_metrics(hum_prop, model_prop),
    })
T6_model = pd.DataFrame(rows)
print("\nModel-by-model:")
print(T6_model.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

article_indices = {
    article: np.asarray([i for i, s in enumerate(order) if s.startswith(article + '_')])
    for article in ('P1', 'P2', 'P3')
}
rows = []
for article, idx in article_indices.items():
    rows.append({
        'analysis': f'{article} only',
        'n_sentences': len(idx),
        'human_mean_flag_share': hum_prop[idx].mean(),
        'model_mean_flag_share': mod_prop[idx].mean(),
        **ordering_metrics(hum_prop[idx], mod_prop[idx]),
    })
for article, excluded in article_indices.items():
    idx = np.asarray([i for i in range(len(order)) if i not in set(excluded)])
    rows.append({
        'analysis': f'without {article}',
        'n_sentences': len(idx),
        'human_mean_flag_share': hum_prop[idx].mean(),
        'model_mean_flag_share': mod_prop[idx].mean(),
        **ordering_metrics(hum_prop[idx], mod_prop[idx]),
    })
T6_article = pd.DataFrame(rows)
print("\nArticle sensitivity:")
print(T6_article.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

REPORT['rq3'] = {
    'overall': T6_overall.to_dict('records')[0],
    'by_model': T6_model.to_dict('records'),
    'by_article': T6_article.to_dict('records'),
}


## Sensitivity analyses


In [ ]:
def panel_row(label, subset):
    selected, n_valid = model_vec(cons, subset, order)
    prop = selected / n_valid
    return {
        'panel': label,
        'n_models': len(subset),
        'mean_flag_share': prop.mean(),
        **ordering_metrics(hum_prop, prop),
    }

rows = [panel_row('all six', models)]
rows += [
    panel_row(f'minus {model}', [m for m in models if m != model])
    for model in models
]
closed_models = ['claude', 'gemini', 'gpt']
open_models = ['deepseek', 'llama', 'qwen']
rows += [
    panel_row('closed-weight only', closed_models),
    panel_row('open-weight only', open_models),
]
T7_panel = pd.DataFrame(rows)
print(T7_panel.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

rows = []
for category in BIAS:
    hs, hnn = human_vec(cells, order, anns, category)
    ms, mnn = model_vec(cons, models, order, category)
    hp, mp = hs / hnn, ms / mnn
    rows.append({
        'category': category,
        'human_sentence_share': hp.mean(),
        'model_sentence_share': mp.mean(),
        'model_selected_cells': int(ms.sum()),
        **ordering_metrics(hp, mp),
    })
T7_category = pd.DataFrame(rows)
print("\nPer bias category:")
print(T7_category.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# Unconsolidated repetitions, after the same contradiction deletion
rows = []
for rep in (1, 2, 3):
    selected, n_valid = repetition_vec(reps, models, order, rep)
    prop = selected / n_valid
    rows.append({
        'repetition': rep,
        'valid_model_sentence_cells': int(n_valid.sum()),
        'mean_flag_share': prop.mean(),
        **ordering_metrics(hum_prop, prop),
    })
T7_repetition = pd.DataFrame(rows)
print("\nEach cleaned repetition alone:")
print(T7_repetition.to_string(index=False, float_format=lambda x: f"{x:.3f}"))



REPORT['rq3_sensitivity'] = {
    'panel': T7_panel.to_dict('records'),
    'category': T7_category.to_dict('records'),
    'repetition': T7_repetition.to_dict('records'),
}


## F1, F2

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(13,5))
jit=np.random.default_rng(SEED).normal(0,.006,57)
ax[0].scatter(hum_prop,mod_prop+jit,s=42,alpha=.75,edgecolor='k',linewidth=.4)
xs=np.linspace(0,1,200); ax[0].plot(xs,1/(1+np.exp(-(b[0]+b[1]*xs))),lw=2,label='P(model flags)')
ax[0].plot([0,1],[0,1],ls=':',c='grey',label='parity'); ax[0].axhline(.5,ls='--',c='r',lw=.8)
for s in ('P1_3','P3_9'):
    i=order.index(s); ax[0].annotate(s,(hum_prop[i],mod_prop[i]),textcoords='offset points',xytext=(6,6),fontsize=9)
ax[0].set_xlabel('proportion of valid readers flagging'); ax[0].set_ylabel('proportion of models flagging')
ax[0].set_title('F1  Level and ordering'); ax[0].legend(fontsize=8)
ax[1].bar(range(30),T5.sort_values('pct_flagged').pct_flagged.values,color='steelblue')
for m in models:
    v=[cons[(m,s)] for s in order if cons[(m,s)] is not None]
    ax[1].axhline(100*sum(1 for x in v if x!=frozenset({'NB'}))/len(v),ls='--',lw=.8,c='crimson')
ax[1].set_xlabel('annotator (sorted)'); ax[1].set_ylabel('% of valid sentences flagged')
ax[1].set_title('F2  Human heterogeneity (red dashed = models)')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'F1_F2.png',dpi=200); plt.show()

## Export



In [ ]:
report_path = OUTPUT_DIR / "analysis_report.json"
with report_path.open("w", encoding="utf-8") as report_file:
    json.dump(REPORT, report_file, ensure_ascii=False, indent=2, default=str)

tables = {
    'T2_model_summary': T2_model,
    'T2_category_sentence_shares': T2_category,
    'T2_category_label_shares': T2_label_share,
    'T2_within_bias_label_profile': T2_label_profile,
    'T3_agreement': T3,
    'T4_six_human_panel_comparison': T4,
    'T5_annotator_heterogeneity': T5.reset_index(),
    'T6_overall': T6_overall,
    'T6_by_model': T6_model,
    'T6_by_article': T6_article,
    'T7_panel_sensitivity': T7_panel,
    'T7_category_sensitivity': T7_category,
    'T7_repetition_sensitivity': T7_repetition,
}
for name, table in tables.items():
    table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

print(f"Wrote report, tables, and figures to {OUTPUT_DIR}")

if CREATE_ZIP:
    import shutil
    archive_base = PROJECT_DIR / "analysis_outputs"
    archive_path = shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=OUTPUT_DIR,
    )
    print(f"ZIP archive (outside output folder): {archive_path}")
else:
    print("ZIP creation is disabled (CREATE_ZIP=False).")
